# 0.import

In [1]:
import os, time, gc, torch, tqdm, numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.utils import negative_sampling
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import GradScaler
from neo4j.exceptions import ServiceUnavailable
from neomodel import config as neomodel_config, db
import gc

### Connect Neo4j 

In [2]:
from app.config.local import (NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD)
neomodel_config.DATABASE_URL = f'bolt://{NEO4J_USER}:{DB_PASSWORD}@{NEO4J_HOST}:{NEO4J_PORT}'
try:
    result, _ = db.cypher_query("RETURN 1 AS test")
    print("✅ Neo4j 연결 성공! → 응답값:", result[0][0])
except ServiceUnavailable as e:
    print("❌ Neo4j 연결 실패:", e)

✅ config_module: ['CARD_SERVER_ADDRESS', 'DB_HOST', 'DB_NAME', 'DB_PASSWORD', 'DB_PORT', 'DB_USER', 'DEBUG', 'GEMINI_API_KEY', 'GOOGLE_APPLICATION_CREDENTIALS', 'GOOGLE_PROJECT', 'NEO4J_HOST', 'NEO4J_PORT', 'NEO4J_USER', 'SWAGGER_SPECS_JSON_ROUTE', 'SWAGGER_SPECS_ROUTE', 'SWAGGER_STATIC_PATH', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os']
✅ Neo4j 연결 성공! → 응답값: 1


### Hyperparameters

In [3]:
EPOCHS      = 100
CLIP_NORM   = 1.0
NEG_K       = 10         # positive 당 negative 100개 생성
WEG_K = 20 # 양성 샘플의 가중치를 2배로 취급.

PATIENCE    = 20
TOP_K       = 5
INIT_LR     = 1e-3
BATCH_SIZE  = 1_024
VAL_BATCH   = 10_000
NUM_MEMBER  = 15  # number of neighbor sample
NUM_ORDER   = 5  # number of neighbor's order 


device = 'cuda' if torch.cuda.is_available() else 'cpu'
POS_W       = torch.tensor(WEG_K, device=device)   # ② BCE pos_weight = NEG_K, 양성 샘플에 부여할 가중치

In [4]:
CKPT_DIR    = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

In [5]:
print("✅ 하이퍼파라미터 설정 완료")

✅ 하이퍼파라미터 설정 완료


### Helper function

In [6]:
# implicit casting at every time
def cypher_df(q: str, params=None) -> pd.DataFrame:
    rec, cols = db.cypher_query(q, params or {})
    df = pd.DataFrame(rec, columns=cols)
    id_cols = [c for c in df.columns if c.endswith('_id') or c in {'mid','oid','pid'}]
    for col in id_cols:
        df[col] = df[col].astype(str)  # 명확히 문자열로 변환 (product_id 포함)
    return df

def safe_map(series, mapping, col):
    mapped = series.map(mapping)
    miss = mapped.isna().sum()
    if miss:
        print(f"⚠️  {col}: {miss:,} unmapped → drop")
        mapped = mapped.dropna()
    return mapped.astype(np.int64).to_numpy()

<hr>

# 1. Data preprocessing

### Load data from neo4j

In [7]:
df_member  = cypher_df("MATCH (m:Member)  RETURN m.member_id  AS mid")
df_order   = cypher_df("MATCH (o:Order)   RETURN o.order_id   AS oid")
df_product = cypher_df("MATCH (p:Product) RETURN p.product_id AS pid")

# mapping
m2i = {mid: i for i, mid in enumerate(df_member.mid.unique())}
o2i = {oid: i for i, oid in enumerate(df_order.oid.unique())}
p2i = {pid: i for i, pid in enumerate(df_product.pid.unique())}

# reverse mapping
i2m = {i: mid_str for mid_str, i in m2i.items()}

# 검증 로직 추가 - 타입 확인
sample_pid = next(iter(p2i.keys()))
assert isinstance(sample_pid, str), f"❌ product_id({sample_pid}) is not str."

df_order_feat = cypher_df("""
MATCH (o:Order)
RETURN o.order_id AS oid,
       o.days_since_prior_order_norm AS dsp,
       o.order_dow_norm              AS dow,
       o.order_hour_of_day_norm      AS hr,
       o.order_number_norm           AS idx
""")
df_order_feat["oid_idx"] = safe_map(df_order_feat.oid, o2i, "oid")
order_feat = (df_order_feat
                .sort_values("oid_idx")[["dsp","dow","hr","idx"]]
                .to_numpy(np.float32))

df_prod_vec = cypher_df("""
MATCH (p:Product)
RETURN p.product_id AS pid,
       p.category_vector + p.name_vector AS vec
""")
df_prod_vec["pid_idx"] = safe_map(df_prod_vec.pid, p2i, "pid")
prod_feat = torch.tensor(
    np.stack(df_prod_vec.sort_values("pid_idx").vec.values),
    dtype=torch.float32
)

### Create edges

In [8]:
# Train Data
df_edge_tr = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.role_train='prior'
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")
uo = torch.stack((torch.from_numpy(safe_map(df_edge_tr.mid, m2i, "mid")),
                  torch.from_numpy(safe_map(df_edge_tr.oid, o2i, "oid"))))
op = torch.stack((torch.from_numpy(safe_map(df_edge_tr.oid, o2i, "oid")),
                  torch.from_numpy(safe_map(df_edge_tr.pid, p2i, "pid"))))

df_label_tr = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.role_train='train'
RETURN m.member_id AS mid, p.product_id AS pid
""")
pos_u = torch.from_numpy(safe_map(df_label_tr.mid, m2i, "mid"))
pos_p = torch.from_numpy(safe_map(df_label_tr.pid, p2i, "pid"))
edge_pos = torch.stack((pos_u, pos_p))

edge_neg = negative_sampling(
    edge_index=edge_pos,
    num_nodes=(len(m2i), len(p2i)),
    num_neg_samples=edge_pos.size(1) * NEG_K
)
edge_label_index = torch.cat([edge_pos, edge_neg], 1)
edge_label = torch.cat([torch.ones(edge_pos.size(1)),
                        torch.zeros(edge_neg.size(1))])

In [9]:
# Test Data 준비 (검증 단계)
df_edge_te = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='prior'
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")
uo_te = torch.stack((torch.from_numpy(safe_map(df_edge_te.mid,m2i,"mid")),
                     torch.from_numpy(safe_map(df_edge_te.oid,o2i,"oid"))))
op_te = torch.stack((torch.from_numpy(safe_map(df_edge_te.oid,o2i,"oid")),
                     torch.from_numpy(safe_map(df_edge_te.pid,p2i,"pid"))))



### Create gt_dict

In [10]:
# GT 데이터 정확히 변환 후 검증
df_gt = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='train'
RETURN m.member_id AS mid, collect(p.product_id) AS gt_list
""")

mid_arr = safe_map(df_gt.mid, m2i, "mid")

# 추가 검증로직 (명확한 변환, unmapped 처리)
gt_list_arr = []
for idx, pid_list in enumerate(df_gt.gt_list):
    mapped_list = [p2i[pid] for pid in pid_list if pid in p2i]
    if not mapped_list:
        print(f"⚠️ unmapped product in gt_list at index {idx}")
    gt_list_arr.append(mapped_list)

gt_dict = {
    int(mid): [int(pid) for pid in pid_list]
    for mid, pid_list in zip(mid_arr, gt_list_arr) if mid != -1 and pid_list
}

### Validate

In [11]:
# 최종 검증 출력
print(f"✅ GT users (index): {len(gt_dict):,}")
sample_gt_key = next(iter(gt_dict.keys()))
assert isinstance(sample_gt_key, int), f"❌ gt_dict key {sample_gt_key} not int."
assert all(isinstance(pid, int) for pid in gt_dict[sample_gt_key]), "❌ product idx not int in gt_dict."
print(f"✅ sample GT for user {sample_gt_key}: {gt_dict[sample_gt_key][:5]}")

✅ GT users (index): 75,000
✅ sample GT for user 189903: [33035, 7323, 3421, 36014, 13382]


### Create HectorData

In [12]:
data = HeteroData()
data['user'].x    = nn.Embedding(len(m2i), 32).weight
data['order'].x   = order_feat
data['product'].x = prod_feat
data['user'].num_nodes = len(m2i)
data['order'].num_nodes= len(o2i)
data['product'].num_nodes=len(p2i)
data['user','ordered','order'].edge_index        = uo
data['order','ordered_by','user'].edge_index     = uo.flip(0)
data['order','contains','product'].edge_index    = op
data['product','contained_in','order'].edge_index= op.flip(0)
data['user','buys','product'].edge_index         = edge_label_index

loader = LinkNeighborLoader(
    data,
    edge_label_index=(('user','buys','product'), edge_label_index),
    edge_label=edge_label,
    batch_size=BATCH_SIZE,
    num_neighbors=[15,5],
    shuffle=True
)

In [13]:
def build_subgraph_for_users(user_ids,  # 실제 사용자 문자열 ID 리스트
                             uo_te, op_te,
                             order_feat, prod_feat,
                             m2i, o2i, p2i,
                             gt_dict):
    # 1) user_ids (문자열 ID)를 m2i를 통해 글로벌 정수 인덱스로 변환
    # user_idx_array는 현재 배치 사용자들의 글로벌 정수 인덱스 배열
    user_idx_array = np.array([m2i[u_str_id] for u_str_id in user_ids if u_str_id in m2i], dtype=np.int64)

    # 만약 user_ids의 모든 사용자가 m2i에 없다면 user_idx_array가 비게 됨
    if len(user_idx_array) == 0:
        # 빈 HeteroData와 빈 gt_local 반환
        empty_data = HeteroData()
        empty_data['user'].num_nodes = 0
        empty_data['order'].num_nodes = 0
        empty_data['product'].num_nodes = 0
        empty_data['user'].x = torch.empty((0, 32)) # 예시 차원, 모델에 맞게 조정
        empty_data['order'].x = torch.empty((0, order_feat.shape[1] if order_feat.ndim > 1 and order_feat.shape[0] > 0 else 0))
        empty_data['product'].x = torch.empty((0, prod_feat.shape[1] if prod_feat.ndim > 1 and prod_feat.shape[0] > 0 else 0))
        return empty_data, {}

    # 2) prior 엣지 기반 order/product 수집
    # uo_te[0]는 이미 글로벌 정수 사용자 인덱스를 담고 있음
    mask_uo   = np.isin(uo_te[0].cpu().numpy(), user_idx_array)
    uo_sub    = uo_te[:, mask_uo]
    order_idx = np.unique(uo_sub[1].cpu().numpy())

    # op_te[0]는 이미 글로벌 정수 주문 인덱스를 담고 있음
    mask_op   = np.isin(op_te[0].cpu().numpy(), order_idx)
    op_sub    = op_te[:, mask_op]
    # op_sub[1]는 글로벌 정수 상품 인덱스
    product_idx_set = set(op_sub[1].cpu().numpy())

    # 3) product_idx_set에 ground truth 상품 추가
    for original_member_id_str in user_ids:
        if original_member_id_str in m2i: # m2i에 해당 사용자가 있는 경우에만 처리
            global_member_idx = m2i[original_member_id_str]
            if global_member_idx in gt_dict:
                product_idx_set.update(gt_dict[global_member_idx])

    product_idx_array = np.fromiter(product_idx_set, dtype=np.int64)

    # 4) 전역 -> 로컬 인덱스 매핑
    old2new_user  = {old_global_idx: i for i, old_global_idx in enumerate(user_idx_array)}
    old2new_order = {old_global_idx: i for i, old_global_idx in enumerate(order_idx)}
    old2new_prod  = {old_global_idx: i for i, old_global_idx in enumerate(product_idx_array)}

    # 5) edge_index 리매핑
    # uo_sub[0]의 요소는 이미 글로벌 정수 사용자 인덱스
    # 해당 인덱스가 old2new_user에 있는지 확인 (user_idx_array에 포함된 사용자만 고려)
    uo_remap_src_list = []
    uo_remap_dst_list = []
    for i in range(uo_sub.size(1)):
        u_global_idx = uo_sub[0, i].item()
        o_global_idx = uo_sub[1, i].item()
        if u_global_idx in old2new_user and o_global_idx in old2new_order: # 서브그래프에 포함된 노드들만
            uo_remap_src_list.append(old2new_user[u_global_idx])
            uo_remap_dst_list.append(old2new_order[o_global_idx])

    if len(uo_remap_src_list) > 0:
        uo_remap = torch.tensor([uo_remap_src_list, uo_remap_dst_list], dtype=torch.long)
    else:
        uo_remap = torch.empty((2,0), dtype=torch.long)


    op_remap_src_list = []
    op_remap_dst_list = []
    for i in range(op_sub.size(1)):
        o_global_idx = op_sub[0, i].item()
        p_global_idx = op_sub[1, i].item()
        if o_global_idx in old2new_order and p_global_idx in old2new_prod: # 서브그래프에 포함된 노드들만
            op_remap_src_list.append(old2new_order[o_global_idx])
            op_remap_dst_list.append(old2new_prod[p_global_idx])

    if len(op_remap_src_list) > 0:
        op_remap = torch.tensor([op_remap_src_list, op_remap_dst_list], dtype=torch.long)
    else:
        op_remap = torch.empty((2,0), dtype=torch.long)


    # 6) 노드 feature
    # 비어있는 경우 처리 추가
    order_x   = torch.tensor(order_feat[order_idx], dtype=torch.float32) if len(order_idx) > 0 else torch.empty((0, order_feat.shape[1] if order_feat.ndim > 1 and order_feat.shape[0] > 0 else 0), dtype=torch.float32)
    product_x = prod_feat[product_idx_array] if len(product_idx_array) > 0 else torch.empty((0, prod_feat.shape[1] if prod_feat.ndim > 1 and prod_feat.shape[0] > 0 else 0), dtype=torch.float32)


    # 7) 로컬 GT 생성
    gt_local = {}
    for original_member_id_str in user_ids:
        if original_member_id_str in m2i: # m2i에 해당 사용자가 있는 경우에만 처리
            global_member_idx = m2i[original_member_id_str]
            if global_member_idx in gt_dict and global_member_idx in old2new_user: # gt가 있고, 서브그래프의 사용자인 경우
                local_user_subgraph_idx = old2new_user[global_member_idx]
                gt_global_product_indices = gt_dict[global_member_idx]
                local_gt_product_indices = []
                for global_prod_idx in gt_global_product_indices:
                    if global_prod_idx in old2new_prod:
                        local_gt_product_indices.append(old2new_prod[global_prod_idx])
                if local_gt_product_indices:
                    gt_local[local_user_subgraph_idx] = local_gt_product_indices

    # 8) HeteroData 생성
    d = HeteroData()
    d['user'].num_nodes    = len(user_idx_array) # old2new_user의 크기와 동일
    d['order'].num_nodes   = len(old2new_order)  # len(order_idx)와 동일
    d['product'].num_nodes = len(old2new_prod) # len(product_idx_array)와 동일

    # 임베딩 레이어는 노드 수가 0일 때 오류 발생 가능하므로 조건 처리
    d['user'].x    = nn.Embedding(len(user_idx_array), 32).weight if len(user_idx_array) > 0 else torch.empty((0, 32))
    d['order'].x   = order_x
    d['product'].x = product_x

    d['user','ordered','order'].edge_index       = uo_remap
    d['order','ordered_by','user'].edge_index    = uo_remap.flip(0)
    d['order','contains','product'].edge_index   = op_remap
    d['product','contained_in','order'].edge_index = op_remap.flip(0)

    return d, gt_local

@torch.no_grad()
def validate_minibatch(model,
                       user_ids: np.ndarray, # 실제 사용자 문자열 ID 배열
                       uo_te, op_te,
                       order_feat, prod_feat,
                       m2i, o2i, p2i, gt_dict,
                       TOP_K: int = 5,
                       SUB_SIZE: int = 512,
                       CHUNK: int = 200_000):
    """
    user_ids 전체를 SUB_SIZE 단위로 나눠 GPU forward.
    GPU 메모리를 SUB_SIZE 규모(≈ 수백 MB) 안으로 고정할 수 있다.
    """
    model.eval()
    precs = []
    # 'device' 변수가 이 함수가 호출되는 스코프에 정의되어 있어야 합니다.
    # 예: device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if len(user_ids) == 0:
        return 0.0

    for s in range(0, len(user_ids), SUB_SIZE):
        tgt_users = user_ids[s:s+SUB_SIZE] # 현재 미니배치의 실제 사용자 문자열 ID

        # (1) 부분 그래프 생성
        g_sub, gt_sub = build_subgraph_for_users(
            tgt_users, uo_te, op_te,
            order_feat, prod_feat,
            m2i, o2i, p2i, gt_dict
        )

        # gt_sub가 비어있으면 (현재 배치에 GT를 가진 사용자가 없으면) 다음 배치로
        if not gt_sub:
            if hasattr(g_sub, 'to'): # g_sub가 생성된 경우 명시적 메모리 해제 시도
                del g_sub
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            continue

        # (2) GPU forward
        # 이 부분은 build_subgraph_for_users에서 빈 그래프를 반환했을 때를 대비해야 함
        if g_sub['user'].num_nodes == 0 : # 사용자가 없는 서브그래프는 스킵
            del g_sub
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue

        g_sub = g_sub.to(device, non_blocking=True)
        z = model(g_sub.x_dict, g_sub.edge_index_dict)
        uemb, pemb = z['user'].cpu(), z['product'].cpu() # GPU에서 CPU로 복사

        # (3) Precision@K 계산
        for uid_local, gt_local_prods in gt_sub.items(): # uid_local은 서브그래프 내 로컬 사용자 인덱스
            if not gt_local_prods: # 해당 사용자의 GT 상품 리스트가 비어있으면 스킵
                continue

            u_embedding = uemb[uid_local]
            
            # pemb (전체 상품 임베딩) 크기가 0인 경우 처리
            if pemb.size(0) == 0:
                precs.append(0.0) # 예측할 상품이 없으므로 precision 0
                continue

            best_scores, best_indices_global = [], []
            for i in range(0, pemb.size(0), CHUNK):
                scores, indices_in_chunk = (pemb[i:i+CHUNK] @ u_embedding).topk(min(TOP_K, pemb[i:i+CHUNK].size(0))) # CHUNK 내 상품수가 TOP_K보다 작을 수 있음
                best_scores.append(scores)
                best_indices_global.append(indices_in_chunk + i) # 원래 상품 인덱스로 변환

            if not best_scores: # 만약 pemb가 비어서 best_scores가 비었다면
                 precs.append(0.0)
                 continue

            best_scores = torch.cat(best_scores)
            best_indices_global = torch.cat(best_indices_global)

            # 전체 상품 중에서 TOP_K개 선택
            # best_scores의 길이가 TOP_K보다 작을 수 있으므로 min 사용
            num_to_select = min(TOP_K, best_scores.size(0))
            if num_to_select == 0: # 선택할 추천 상품이 없으면
                precs.append(0.0)
                continue

            final_top_scores, final_top_indices_in_best = best_scores.topk(num_to_select)
            top_pred_local_prod_indices = best_indices_global[final_top_indices_in_best].tolist()

            hit = len(set(top_pred_local_prod_indices) & set(gt_local_prods))
            precs.append(hit / TOP_K) # Precision@K (K로 나누는 것이 표준)

        # (4) 메모리 반환
        del g_sub, z, uemb, pemb
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return float(np.mean(precs)) if precs else 0.0

In [14]:
loader = LinkNeighborLoader(
    data,
    edge_label_index=(('user','buys','product'), edge_label_index),
    edge_label=edge_label,
    batch_size=BATCH_SIZE,
    num_neighbors=[NUM_MEMBER,NUM_ORDER],
    shuffle=True
)

<hr>

# 2. Model setup

In [15]:
class HetSAGE(nn.Module):
    def __init__(self, hid=128, out=128, p_drop=0.2, aggr='sum'):
        super().__init__()
        self.conv1 = HeteroConv({
            ('user','ordered','order')   : SAGEConv((-1,-1), hid, aggr=aggr),
            ('order','contains','product'): SAGEConv((-1,-1), hid, aggr=aggr),
            ('order','ordered_by','user') : SAGEConv((-1,-1), hid, aggr=aggr),
            ('product','contained_in','order'): SAGEConv((-1,-1), hid, aggr=aggr),
        }, aggr=aggr)
        self.conv2 = HeteroConv({
            ('user','ordered','order')   : SAGEConv((hid,hid), out, aggr=aggr),
            ('order','contains','product'): SAGEConv((hid,hid), out, aggr=aggr),
            ('order','ordered_by','user') : SAGEConv((hid,hid), out, aggr=aggr),
            ('product','contained_in','order'): SAGEConv((hid,hid), out, aggr=aggr),
        }, aggr=aggr)
        self.norm1, self.norm2 = nn.LayerNorm(hid), nn.LayerNorm(out)
        self.p_drop = p_drop
    def _drop(self,h): return {k:F.dropout(v,p=self.p_drop,training=self.training) for k,v in h.items()}
    def forward(self,x_dict,edge_index_dict):
        h=self.conv1(x_dict,edge_index_dict); h={k:F.relu(self.norm1(v)) for k,v in h.items()}; h=self._drop(h)
        h=self.conv2(h,edge_index_dict);      h={k:F.relu(self.norm2(v)) for k,v in h.items()}; h=self._drop(h)
        return h

model = HetSAGE().to(device)

<hr>

# 3. Training

### Memory Freeing

In [16]:
# 학습 시작 직전 메모리 해제 (OOM 방지)
def cleanup_memory(*vars_to_delete):
    for var in vars_to_delete:
        if var in globals():
            del globals()[var]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

cleanup_memory(
    'df_member', 'df_order', 'df_product',
    'df_order_feat', 'df_prod_vec',
    'df_edge_tr', 'df_label_tr',
    'df_edge_te', 'df_gt',
    'pos_u', 'pos_p', 'edge_pos', 'edge_neg',
    'mid_arr', 'gt_list_arr'
)

print("✅ 불필요한 메모리 자원 정리 완료!")

✅ 불필요한 메모리 자원 정리 완료!


In [17]:
# 지정한 텐서/변수를 삭제하고 CUDA 캐시 비움
def free_gpu(*tensors):
    for t in tensors:
        del t
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [18]:
opt       = torch.optim.Adam(model.parameters(), lr=INIT_LR, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
scaler    = GradScaler()

history   = {"epoch": [], "train_loss": [], "val_prec": []}
BEST_PREC, no_improve = 0.0, 0
test_user_ids = [i2m[idx] for idx in gt_dict.keys() if idx in i2m]

for ep in range(1, EPOCHS + 1):
    tic = time.perf_counter()
    model.train(); running_loss = seen = 0.0

    pbar = tqdm.tqdm(loader, desc=f"[Train] {ep}/{EPOCHS}", ncols=100)
    for batch in pbar:
        # ─────────────── 학습 step ────────────────
        batch = batch.to(device, non_blocking=True)
        bs = batch['user','buys','product'].edge_label.size(0)

        with torch.amp.autocast(device_type='cuda'):
            z   = model(batch.x_dict, batch.edge_index_dict)
            src = z['user'   ][batch['user','buys','product'].edge_label_index[0]]
            dst = z['product'][batch['user','buys','product'].edge_label_index[1]]
            logits = (src * dst).sum(-1)
            loss   = F.binary_cross_entropy_with_logits(logits,
                                                        batch['user','buys','product'].edge_label,
                                                        pos_weight=POS_W)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)

        running_loss += loss.item() * bs;  seen += bs
        pbar.set_postfix(loss=f"{running_loss/seen:.4f}")

        # ─── 미니-배치 메모리 반환 ───
        free_gpu(batch, z, src, dst, logits, loss)

    train_loss = running_loss / seen

    # ─────────────── 검증 step ────────────────
    np.random.seed(ep)
    val_users = np.random.choice(test_user_ids, VAL_BATCH, replace=False)

    val_prec = validate_minibatch(model, val_users,
                                  uo_te, op_te,
                                  order_feat, prod_feat,
                                  m2i, o2i, p2i,
                                  gt_dict,
                                  TOP_K=TOP_K,
                                  SUB_SIZE=512)        # SUB_SIZE로 OOM 방지

    print(f"✅ Epoch {ep} | loss={train_loss:.4f} | P@{TOP_K}={val_prec:.4f} | "
          f"{time.perf_counter()-tic:.1f}s")

    # ───────── 스케줄러 & 체크포인트 ──────────
    scheduler.step(val_prec)
    if val_prec > BEST_PREC:
        BEST_PREC, no_improve = val_prec, 0
        torch.save(
            {"epoch": ep, "state": model.state_dict(),
             "m2i": m2i, "o2i": o2i, "p2i": p2i, "precision": BEST_PREC},
            os.path.join(CKPT_DIR, "best_model_v1.pt"))
        print("💾  New best model saved")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("🛑  Early-stopping triggered"); break

    # ─── epoch-단위 메모리 반환 ───
    free_gpu(val_users)

    # ─── 히스토리 저장 ───
    history["epoch"].append(ep)
    history["train_loss"].append(train_loss)
    history["val_prec"].append(val_prec)

print("🎯 Training complete!")

[Train] 1/100: 100%|███████████████████████████████| 8275/8275 [26:51<00:00,  5.13it/s, loss=1.5387]


✅ Epoch 1 | loss=1.5387 | P@5=0.0018 | 1627.5s
💾  New best model saved


[Train] 2/100: 100%|███████████████████████████████| 8275/8275 [26:49<00:00,  5.14it/s, loss=1.4941]


✅ Epoch 2 | loss=1.4941 | P@5=0.0020 | 1625.6s
💾  New best model saved


[Train] 3/100: 100%|███████████████████████████████| 8275/8275 [26:53<00:00,  5.13it/s, loss=1.5030]


...

✅ Epoch 32 | loss=1.2610 | P@5=0.0245 | 1636.5s


[Train] 33/100: 100%|██████████████████████████████| 8275/8275 [26:59<00:00,  5.11it/s, loss=1.2603]


✅ Epoch 33 | loss=1.2603 | P@5=0.0219 | 1635.5s


[Train] 34/100: 100%|██████████████████████████████| 8275/8275 [26:58<00:00,  5.11it/s, loss=1.2569]


✅ Epoch 34 | loss=1.2569 | P@5=0.0282 | 1629.9s
🛑  Early-stopping triggered
🎯 Training complete!


<hr>

import json

# ... (기존 훈련 코드 끝부분) ...
print("🎯 Training complete!")

# history 딕셔너리를 JSON 파일로 저장
with open('training_history.json', 'w') as f:
    json.dump(history, f)

print("💾 Training history saved to training_history.json")

# 4. Visualization

In [1]:
import json
import matplotlib.pyplot as plt

# 저장된 history 파일 불러오기
with open('training_history.json', 'r') as f:
    history = json.load(f)

print("✅ Training history loaded.")

✅ Training history loaded.


In [2]:
df = pd.DataFrame(history)
plt.figure()
plt.plot(df["epoch"], df["train_loss"], label="Train loss")
plt.plot(df["epoch"], df["val_prec"], label=f"P@{TOP_K}")
plt.xlabel("Epoch"); plt.legend(); plt.title("Training history"); plt.grid(True)
plt.show()

NameError: name 'pd' is not defined